In [ ]:
import numpy as np
from pymodbus.client import ModbusTcpClient
from socket import create_connection
from time import sleep
import os
HEAT_HOST = os.getenv('HEAT_HOST', 'localhost')
HEAT_PORT = int(os.getenv('HEAT_PORT', '502'))
PTM_HOST = os.getenv('PTM_HOST', 'localhost')
PTM_PORT = int(os.getenv('PTM_PORT', '502'))
SENSOR_HOST = os.getenv('SENSOR_HOST', 'localhost')
SENSOR_PORT = int(os.getenv('SENSOR_PORT', '9090'))
heat = ModbusTcpClient(HEAT_HOST, port=HEAT_PORT)
heat.connect()
heat.write_register(801, 12000)
ptm = ModbusTcpClient(PTM_HOST, port=PTM_PORT)
ptm.connect()
sensr = create_connection((SENSOR_HOST, SENSOR_PORT))
X = np.memmap(os.getenv('DISK_DATA', 'real4.47f,34f'), '47f,34f', shape=9999)
Y = X['f1']
X = X['f0']
I = np.array(Y.any(1).argmin())
for i in range(I, 7):
    F(np.array([2*(j==i)for j in range(47)]))
if I==7:
    F(np.zeros(47))

In [ ]:
from scipy.optimize import lsq_linear, nnls
def F(x, t=10, eps=.025):
    print(x)
    X[I] = x
    heat.write_registers(5, (x[:7]*100).astype('H').tolist())
    ptm.write_registers(4085, (x[7:]*400).astype('H').tolist())
    ptm.write_register(4211, 65535)
    y = []
    while len(y)<t or (np.max(y[-t:],0)-np.min(y[-t:],0)).max()>eps:
        sleep(1.1)
        sensr.send(b'GetData')
        y += [np.array(sensr.recv(999)[:-1].split(b',')[-34:],'f')]
    print(len(y), y[-1].mean(), 3*y[-1].std())
    Y[I] = y[-1]-120.1
    I[()] += 1
    return Y[I-1]
class Jac:
    def __init__(s, X, Y, x, y, nn=0, inv=0, k=.5):
        if inv:
            X,Y,x,y = Y,X,y,x
        s._ = np.exp(np.linalg.norm(X-x,axis=1,keepdims=1)**k/-.5**k)
        s._ = nnls((X-x)*s._,((Y-y)*s._).astype('d')).x if nn else np.linalg.lstsq((X-x)*s._,(Y-y)*s._)[0]
        if not inv:
            s._ = s._.T
    def __call__(s, x, y, lsq=0, mi=0, ma=3):
        return lsq_linear(s._,y.astype('d'),(x-ma,x-mi)).x if lsq else y@s._
    def step(s, dx, dy, inv=0, eps=1e-9):
        s._ += (s._@dx)[:,None]*(dx-dy@s._)/(dy@s._@dx+eps)if inv else(dy-s._@dx)[:,None]*dx/(np.linalg.norm(dx)+eps)
class Line:
    def __init__(s, _=1):
        s._ = _
    def __call__(s, x, g, y, J, eps=1e-9):
        g = g/(np.linalg.norm(g)+eps)
        _ = min(y@J@g/(np.linalg.norm(J@g)+eps),s._)if s._ else 1
        print('line k', _, ':', np.linalg.norm(y), '->', np.linalg.norm(y-J@(_*g)))
        print(np.linalg.norm(_*g))
        return -_*g
    def step(s, dx, y, J, l, dl):
        k = (np.linalg.norm(y+J@dx)-l)/dl
        _ = s._*2 if k>.75 else s._*.25 if k<.25 else s._
        b = k>.25
        print('line', k, ':', s._, '->', _, b)
        s._ = _
        return b
class Jit:
    def __init__(s, _=.1):
        s._ = _
    def __call__(s, x):
        return x+s._*np.random.randn(len(x))*x.clip(.1)
    def step(s, dl):
        _ = s._*.96
        print('jit', s._, '->', _)
        s._ = _
def root(f, X, Y, I=None, mi=0, ma=3, epoch=128, it=1, k=.5, nn=0, inv=0, inv2=0, lsq=1, line=0, jit=0):
    i = np.linalg.norm(Y[:I], axis=1).argmin()
    x = X[i]
    y = Y[i]
    l = np.linalg.norm(y)
    print('los prev', l)
    line = Line(line)
    jit = Jit(jit)
    for i in range(epoch):
        J = Jac(X[:I], Y[:I], x, y, nn=nn, inv=inv, k=k)
        for j in range(it):
            g = J(x, y, lsq=lsq)
            x_ = jit(line(x, g, y, J._)).clip(mi, ma)
            y_ = f(x_)
            l_ = np.linalg.norm(y_)
            print('los', l_)
            J.step(x_-x, y_-y, inv=inv2)
            jit.step(l_-l)
            if line.step(x_-x, y, J._, l, l_-l):
                x,y,l = x_,y_,l_
    return l

root(F, X, Y, I, epoch=20, it=1)

x = X[np.linalg.norm(Y[:I], axis=1).argmin()].copy()
for i in range(40):
    if i in (3,4,13,14,35,36):
        continue
    x[7+i] += .5
    F(x)
    x[7+i] -= .5

(abs(Y[I-40:I]-Y[29]).max(1)<.1).nonzero() #3,4,13,14,35,36

In [ ]:
i = np.linalg.norm(Y[:I], axis=1).argmin()
x0 = X[i].copy()
y0 = Y[i].copy()
np.linalg.norm(y0)

In [ ]:
F(x0)

In [ ]:
root(lambda x:F(np.concat([x0[:7],x])), X[:,7:], Y, I, epoch=100, it=1, k=1, nn=0, inv=0, inv2=0, lsq=1, line=.1, jit=0)

---

In [ ]:
np.linalg.norm(y)

In [ ]:
i = np.linalg.norm(Y[:I], axis=1).argmin()
x = X[i]
y = Y[i]

In [ ]:
def lsq1(x, y, J, mi=0, ma=3, k=.5, eps=1e-9):
    kl = [(_:=min(max(x-J@y/(np.linalg.norm(J)**2+eps),mi),ma)-x,np.linalg.norm(y+_*J))for x,J in zip(x,J)]
    i,(k,l) = min(enumerate(kl), key=lambda ikl:ikl[1][1]+np.random.rand()*.01)
    return i,k,l
def mima1(x, y, J, mi=0, ma=3, k=.5, eps=1e-9):
    j = np.stack([y.argmin(),y.argmax()])
    dx = (x-k*y[j,None]/(J[:,j].T+eps)).clip(mi,ma)-x
    y = y+dx[...,None]*J
    n,i = np.unravel_index((y.max(2)**2+y.min(2)**2).argmin(), dx.shape)
    return i, dx[n,i]
# i = np.linalg.norm(Y[:I], axis=1).argmin()
# x = (.386+.462*np.random.randn(47)).clip(0,3)
# y = F(x)
J = Jac(X[:I], Y[:I], x, y)._.T
for _ in range(999):
    i,k,l = lsq1(x, y, J)
    print(i, x[i], k, np.linalg.norm(y), l)
    x[i] = (x[i]+k).clip(0,3)
    y_ = F(x)
    J[i] = (y_-y)/(k+1e-4)
    y = y_